# Step 1
Decide what video you are going to use for this homework, select an object and generate the template. You can use any video you want (your own, from Youtube, etc.)
and track any object you want (e.g. a car, a pedestrian, etc.).

In [ ]:
!pip install yt-dlp
!pip uninstall -y opencv-python opencv-contrib-python
!pip install opencv-contrib-python

In [ ]:
import yt_dlp

def download_video(url, filename='car.mp4'):
    ydl_opts = {
        'format': 'best',
        'outtmpl': filename,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

video_url = "https://youtu.be/SqmMDATbgIE"
download_video(video_url)

In [ ]:
N_FRAMES = 30

# Step 2
Initialize a tracker (e.g. KCF).

In [ ]:
import cv2
import matplotlib.pyplot as plt

tracker = cv2.TrackerCSRT_create()

# Step 3
Run the tracker on the video and the selected object. Run the tracker for around 10-15 frames.

In [ ]:
import cv2

video = cv2.VideoCapture("car.mp4")

for _ in range(30):
    video.grab()

ret, frame = video.retrieve()

In [ ]:
plt.imshow(frame)

In [ ]:
x = 240
y = 155
w = 60
h = 45

In [ ]:
test_frame = frame.copy()

cv2.rectangle(test_frame, (x, y), (x + w, y + h), (255, 255, 0), 2)


cv2.putText(test_frame, "Initial BBox", (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)


plt.imshow(test_frame)

In [ ]:
bbox = (x, y, w, h)
tracker.init(frame, bbox)

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt

folder_name = 'CSRT'
if not os.path.exists(folder_name):
    os.makedirs(folder_name)


for i in range(N_FRAMES):
    ret, frame = video.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        p1 = (int(bbox[0]), int(bbox[1]))
        p2 = (int(bbox[0] + bbox[2]), int(bbox[1] + bbox[3]))
        cv2.rectangle(frame, p1, p2, (0, 255, 0), 2, 1)
        cv2.putText(frame, f"Frame: {i+1}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    else:
        cv2.putText(frame, "Tracking failure", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    file_path = os.path.join(folder_name, f"frame_{i:02d}.png")
    cv2.imwrite(file_path, frame)



# Step 5
Select a different tracker (e.g. CSRT) and repeat steps 2, 3 and 4.

In [ ]:
import cv2
import matplotlib.pyplot as plt

tracker = cv2.TrackerMIL_create()

In [ ]:
import cv2

video = cv2.VideoCapture("car.mp4")

for _ in range(30):
    video.grab()

ret, frame = video.retrieve()

In [ ]:
bbox = (x, y, w, h)
tracker.init(frame, bbox)

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt

folder_name = 'MIL'
if not os.path.exists(folder_name):
    os.makedirs(folder_name)


for i in range(N_FRAMES):
    ret, frame = video.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        p1 = (int(bbox[0]), int(bbox[1]))
        p2 = (int(bbox[0] + bbox[2]), int(bbox[1] + bbox[3]))
        cv2.rectangle(frame, p1, p2, (0, 255, 0), 2, 1)
        cv2.putText(frame, f"Frame: {i+1}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    else:
        cv2.putText(frame, "Tracking failure", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    file_path = os.path.join(folder_name, f"frame_{i:02d}.png")
    cv2.imwrite(file_path, frame)


# Step 6

In [ ]:
import cv2
import os

def images_to_video(image_folder, output_folder, method_name, fps=10):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder, exist_ok=True)

    images = sorted([img for img in os.listdir(image_folder) if img.endswith(".png")])
    if not images:
        return None

    first_frame = cv2.imread(os.path.join(image_folder, images[0]))
    height, width, _ = first_frame.shape

    video_name = f"{method_name}_tracked.avi"
    full_output_path = os.path.join(output_folder, video_name)

    fourcc = cv2.VideoWriter_fourcc(*'XVID')

    video_writer = cv2.VideoWriter(full_output_path, fourcc, fps, (width, height))

    for image in images:
        frame = cv2.imread(os.path.join(image_folder, image))
        video_writer.write(frame)

    video_writer.release()
    print(f"Video saved: {full_output_path}")
    return full_output_path

# Run
for m in ['CSRT', 'MIL']:
    images_to_video(m, 'Tracked', m)

In [ ]:
!ffmpeg -i Tracked/CSRT_tracked.avi -vcodec libx264 -crf 25 Tracked/CSRT_tracked.mp4 -y
!ffmpeg -i Tracked/MIL_tracked.avi -vcodec libx264 -crf 25 Tracked/MIL_tracked.mp4 -y

In [ ]:
from IPython.display import HTML
from base64 import b64encode

def display_video(video_path):
    video_file = open(video_path, "rb").read()

    data_url = "data:video/mp4;base64," + b64encode(video_file).decode()

    return HTML(f"""
    <video width=600 controls autoplay loop>
          <source src="{data_url}" type="video/mp4">
          Your browser does not support the video tag.
    </video>
    """)

In [ ]:
display_video('car.mp4')

In [ ]:
# Display the converted version
display_video('/content/Tracked/CSRT_tracked.mp4')

In [ ]:
display_video('/content/Tracked/MIL_tracked.mp4')

Compare the results:
* Do you see any differences? If so, what are they?
> Yup, CSRT starts off very well, but it fails to keep up when the car moves rapidly, resulting in it losing part of the vehicle. In contrast, while MIL uses a larger bbox, it successfully captures and tracks the entire car throughout the movement
* Does one tracker perform better than the other? In what way?
> As I mentioned before, for this video MIL performs better. It manages to keep up with the car's movement and keeps the focus on the entire vehicle, even if the bbox is slightly larger than the object itself